<a href="https://colab.research.google.com/github/saradadevivalina/used-cars-data-prep/blob/main/Data_Cleaning_and_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
print("_____Task 1 — Load, Inspect, and Clean_____")

_____Task 1 — Load, Inspect, and Clean_____


In [34]:
import pandas as pd

car_path = '/content/drive/MyDrive/Colab Notebooks/cardekho_dataset.csv'
df = pd.read_csv(car_path)

In [35]:
print(df.shape)
print(df.info())
print(df.isnull().sum() * 100 / len(df))

#Drop rows where selling_price is missing.
df = df.dropna(subset = ['selling_price'])

# Function to clean and convert columns
def clean_and_convert(df,column,unit_to_strip):
    # Strip units and convert to numeric, coercing errors
    df[column] = df[column].astype(str).str.replace(unit_to_strip,'').str.strip()
    df[column] = pd.to_numeric(df[column],errors='coerce')

    # Fill nulls with median
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)

    print(f"Cleaned and converted '{column}'. Filled {df[column].isnull().sum()} nulls with median {median_value}.")
    return df

# Clean 'mileage' column
df = clean_and_convert(df, 'mileage', 'kmpl')
# Clean 'engine' column
df = clean_and_convert(df, 'engine', 'CC')
# Clean 'max_power' column
df = clean_and_convert(df, 'max_power', 'bhp')
# Verify changes
print(df[['mileage', 'engine', 'max_power']].info())

#Remove rows where selling_price is 999999999 or below 10000
df = df[~((df['selling_price'] == 999999999) | (df['selling_price'] < 10000))]

#Drop all duplicate rows.
df = df.drop_duplicates()

#Print df.shape again after cleaning.

print(f"After cleaning printing shape of df {df.shape}")

(15411, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None
Unnamed: 0           0.0
car_name           

In [36]:
print("Task 2 — Encode Categorical Features")

Task 2 — Encode Categorical Features


In [38]:
# Apply label encoding to transmission_type — map Manual → 0 and Automatic → 1.

df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,0,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,0,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,0,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,0,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,0,22.77,1498,98.59,5,570000


In [39]:
df['transmission_type'].value_counts()

,count
transmission_type,
0,12225
1,3186


In [40]:
# Apply one-hot encoding to fuel_type and seller_type using pd.get_dummies() with drop_first=True.

df['fuel_type'].value_counts()

,count
fuel_type,
Petrol,7643
Diesel,7419
CNG,301
LPG,44
Electric,4


In [41]:
df['seller_type'].value_counts()

,count
seller_type,
Dealer,9539
Individual,5699
Trustmark Dealer,173


In [42]:
df = pd.get_dummies(df, columns = ['fuel_type', 'seller_type'], drop_first = True)
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,transmission_type,mileage,engine,max_power,seats,selling_price,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,seller_type_Individual,seller_type_Trustmark Dealer
0,0,Maruti Alto,Maruti,Alto,9,120000,0,19.70,796,46.30,5,120000,False,False,False,True,True,False
1,1,Hyundai Grand,Hyundai,Grand,5,20000,0,18.90,1197,82.00,5,550000,False,False,False,True,True,False
2,2,Hyundai i20,Hyundai,i20,11,60000,0,17.00,1197,80.00,5,215000,False,False,False,True,True,False
3,3,Maruti Alto,Maruti,Alto,9,37000,0,20.92,998,67.10,5,226000,False,False,False,True,True,False
4,4,Ford Ecosport,Ford,Ecosport,6,30000,0,22.77,1498,98.59,5,570000,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,0,19.81,1086,68.05,5,250000,False,False,False,True,False,False
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,0,17.50,1373,91.10,7,925000,False,False,False,True,False,False
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,0,21.14,1498,103.52,5,425000,True,False,False,False,False,False
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,0,16.00,2179,140.00,7,1225000,True,False,False,False,False,False


In [43]:
# Print the column list of the final DataFrame using df.columns.tolist().

df.columns.tolist()


['Unnamed: 0',
 'car_name',
 'brand',
 'model',
 'vehicle_age',
 'km_driven',
 'transmission_type',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price',
 'fuel_type_Diesel',
 'fuel_type_Electric',
 'fuel_type_LPG',
 'fuel_type_Petrol',
 'seller_type_Individual',
 'seller_type_Trustmark Dealer']

Why is drop_first=True used in one-hot encoding?

When we do one-hot encode on a categorical variable with N unique categories, pd.get_dummies() by default creates N new columns (dummy variables). The problem is that these N columns are not independent. If we know the values of N-1 of these columns, we can perfectly predict the value of the remaining Nth column. This perfect linear relationship among predictor variables is called multicollinearity. Multicollinearity can cause issues in certain statistical models (like linear regression) by making the model coefficients unstable and difficult to interpret. By setting drop_first=True, pd.get_dummies() drops one of the N dummy variables (usually the first one alphabetically or based on its internal ordering). This removes the redundancy and prevents multicollinearity.

What does a row of all zeros in the new dummy columns represent?

When we use drop_first=True, the category that was dropped becomes the reference category. If a row has a value of 0 across all the created dummy variables for that feature, it means that the original category for that row was the one that was dropped.

In [44]:
print("Task 3 — Split and Compute Baseline MAE")

Task 3 — Split and Compute Baseline MAE


In [45]:
# Define X (all columns except selling_price) and y (selling_price). Drop any remaining non-numeric columns from X.

# Define target variable y
y = df['selling_price']

# Define features X (all columns except 'selling_price')
X = df.drop('selling_price', axis=1)

# Drop non-numeric and identifying columns from X
X = X.drop(columns=['Unnamed: 0', 'car_name', 'brand', 'model'], errors='ignore')

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("\nFirst 5 rows of X:")
print(X.head())
print("\nFirst 5 values of y:")
print(y.head())

Shape of X: (15411, 13)
Shape of y: (15411,)

First 5 rows of X:
   vehicle_age  km_driven  transmission_type  mileage  engine  max_power  \
0            9     120000                  0    19.70     796      46.30   
1            5      20000                  0    18.90    1197      82.00   
2           11      60000                  0    17.00    1197      80.00   
3            9      37000                  0    20.92     998      67.10   
4            6      30000                  0    22.77    1498      98.59   

   seats  fuel_type_Diesel  fuel_type_Electric  fuel_type_LPG  \
0      5             False               False          False   
1      5             False               False          False   
2      5             False               False          False   
3      5             False               False          False   
4      5              True               False          False   

   fuel_type_Petrol  seller_type_Individual  seller_type_Trustmark Dealer  
0          

In [46]:
# Split into 80% train and 20% test using train_test_split with random_state=42.

from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the resulting datasets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (12328, 13)
Shape of X_test: (3083, 13)
Shape of y_train: (12328,)
Shape of y_test: (3083,)


In [48]:
from sklearn.metrics import mean_absolute_error

# Calculate the mean of y_train
y_train_mean = y_train.mean()
y_pred_baseline = [y_train_mean] * len(y_test)

# Calculate the Mean Absolute Error (MAE)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)

print(f"Mean of y_train: {y_train_mean:.2f}")
print(f"Baseline MAE: ₹{mae_baseline:.0f}")

Mean of y_train: 772120.62
Baseline MAE: ₹468748
